# Entraînement sur Google Colab (GPU)

Ce notebook entraîne les modèles du projet sur le GPU gratuit de Colab, car le PC local n'a pas de GPU utilisable.

**Avant de commencer :** menu *Exécution > Modifier le type d'exécution > GPU T4*.

Ensuite, exécute les cellules **dans l'ordre**, de haut en bas. Rien n'est conservé quand la session Colab se ferme :
pense à télécharger les résultats (dernière cellule).

## 1. Vérifier le GPU

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPU :", tf.config.list_physical_devices("GPU"))   # une liste vide = pas de GPU, change le type d'exécution

## 2. Récupérer le code du projet

On clone le dépôt GitHub : Colab travaille ainsi avec exactement le même code que le PC.
Le dépôt doit être à jour (`git push` depuis le PC) et **public** (sur GitHub : *Settings > General > Change visibility*).

S'il reste privé, remplace la première ligne par les deux lignes en commentaire : elles demandent un jeton GitHub
(*Settings > Developer settings > Personal access tokens*, droit de lecture sur le dépôt).

In [ ]:
!git clone https://github.com/abdellahessou/deeplearning.git
# Dépôt privé : supprime la ligne ci-dessus et enlève le # des deux lignes ci-dessous
# import getpass; jeton = getpass.getpass("Jeton GitHub : ")
# !git clone https://{jeton}@github.com/abdellahessou/deeplearning.git
%cd deeplearning

## 3. Télécharger le dataset Kaggle

Colle ton jeton Kaggle (`KGAT_...`) quand la cellule le demande. Il n'est **pas** enregistré dans le notebook.

In [ ]:
import os, getpass
!pip install -q -U kaggle
os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Jeton Kaggle (KGAT_...) : ")

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p data/raw --unzip
# L'archive contient un dossier en double et un dossier __MACOSX : on ne garde que train / val / test
!rm -rf data/raw/chest_xray/chest_xray data/raw/chest_xray/__MACOSX
!ls data/raw/chest_xray

## 4. Refaire le split train / val / test

Même script et même seed qu'en local : on doit retrouver 4447 / 785 / 624 images.

In [ ]:
!python src/split_data.py

## 5. Entraîner les 3 modèles (étape 4)

Les trois modèles sont entraînés avec **exactement les mêmes réglages** : 15 epochs au maximum avec la base gelée
(phase 1), puis 10 epochs au maximum de fine-tuning (phase 2). L'arrêt anticipé peut raccourcir chaque phase.

Une cellule par modèle : si la session se coupe, les modèles déjà terminés ne sont pas à refaire.
Compte environ 1 à 2 heures au total.

In [ ]:
!python src/train.py --model mobilenetv2 --epochs1 15 --epochs2 10

In [ ]:
!python src/train.py --model resnet50 --epochs1 15 --epochs2 10

In [ ]:
!python src/train.py --model vgg19 --epochs1 15 --epochs2 10

## 6. Regarder les courbes

Le trait vertical en pointillés marque le début du fine-tuning.

In [ ]:
from IPython.display import Image, display
for m in ["mobilenetv2", "resnet50", "vgg19"]:
    display(Image(f"reports/figures/courbes_{m}.png"))

## 7. Télécharger les résultats

On regroupe dans un fichier ZIP les 3 modèles finaux (`.keras`), les historiques (`.csv`) et les courbes
(environ 350 Mo, le téléchargement prend quelques minutes).
Sur le PC : mets les `.keras` et les `.csv` dans `models/`, et les `.png` dans `reports/figures/`.
La comparaison finale (`python src/compare.py`) se lance ensuite sur le PC.

In [ ]:
!zip -j resultats.zip models/mobilenetv2.keras models/resnet50.keras models/vgg19.keras models/*_history.csv reports/figures/courbes_*.png
from google.colab import files
files.download("resultats.zip")